# M3 – Step 8: Request 3 results, final surrogate, M1 re-run (closes M3)

1. **Final hold-out:** score the predictions frozen *before* request 3 was sent (no refit first)
2. Refit on all 30 runs (29 unique points): LOO and repeated 5-fold CV, parity plot. **Headline metric: RMSE.**
3. Final surrogate (cubic RBF) with uncertainty
4. M1 re-run: how far did the optimum move vs FM0, vs the 24-point fit, and vs other surrogates?
5. Noise note, and what a smooth model handles badly

For every section: **prediction before running**, **observation** after.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from m3_surrogate import (load_oracle, average_repeats, make_features, disk_loading, QuadraticSurrogate,
                          RBFSurrogate, GPSurrogate, loo_predictions, loo_table, repeated_kfold,
                          jackknife_band, D_MIN, D_MAX, T_MIN, T_MAX)
ORACLE_DIR = ROOT / 'data' / 'oracle'; RES_DIR = ROOT / 'results'; FIG_DIR = ROOT / 'figures'
pd.set_option('display.width', 200)
raw = load_oracle(ORACLE_DIR)
print(raw.groupby('batch').size().to_string())

## 1. Final hold-out: request 3 vs the frozen predictions

The models were fitted on 23 points and their predictions saved before the oracle answered.

Prediction (which model will do best? which point will surprise?): _..._

In [ ]:
frozen = pd.read_csv(RES_DIR / 'm3_request3_frozen_predictions.csv')
plan3 = pd.read_csv(RES_DIR / 'm3_request3_plan.csv', index_col=0)
r3 = raw[raw.batch == 'result_3'].reset_index(drop=True)
assert np.allclose(frozen[['D_m', 'T_N']].values, r3[['D_m', 'T_N']].values), "result_3 rows do not match request 3"
frozen.index = plan3.index
frozen.insert(2, 'category', plan3.category.values)
frozen.insert(3, 'DL', plan3.DL.round(0).values)
frozen['FM oracle'] = r3.FM.values
model_cols = ['quadratic', 'RBF thin-plate exact', 'RBF cubic', 'GP (ARD + nugget)']
display(frozen.round(3))
e = frozen[model_cols].rsub(frozen['FM oracle'], axis=0)
ho3 = pd.DataFrame({'hold-out RMSE': np.sqrt((e**2).mean()), 'hold-out MAE': e.abs().mean(),
                    'max |error|': e.abs().max(), 'worst point': e.abs().idxmax()})
ho3.to_csv(RES_DIR / 'm3_holdout_request3.csv', float_format='%.4f')
ho3.round(4)

Observation: _..._

## 2. Noise and the refit on all data

Prediction: _..._

In [ ]:
df, repeats = average_repeats(raw)
print(repeats.to_string(index=False))
print(f"noise std estimate ≈ |diff|/√2 = {repeats['diff'].iloc[0] / np.sqrt(2):.4f}  (one pair only)")
X, y = make_features(df.D_m, df.T_N), df.FM.values
models = {'quadratic': QuadraticSurrogate(), 'RBF thin-plate exact': RBFSurrogate('thin_plate_spline', 0.0),
          'RBF cubic': RBFSurrogate('cubic', 0.0), 'GP (ARD + nugget)': GPSurrogate()}
loo = {k: loo_predictions(m, X, y) for k, m in models.items()}
table = loo_table(loo, y)
rows = {}
for k, m in models.items():
    r = repeated_kfold(m, X, y, k=5, repeats=20, seed=0)
    rows[k] = (r.mean(), r.std())
table['5-fold RMSE mean'] = [rows[k][0] for k in table.index]
table['5-fold RMSE std'] = [rows[k][1] for k in table.index]
table['final hold-out RMSE'] = ho3.loc[table.index, 'hold-out RMSE']
table = table[['LOO RMSE', '5-fold RMSE mean', '5-fold RMSE std', 'final hold-out RMSE', 'LOO MAE',
               'max |error|', 'worst pt', 'LOO Q2', 'mean z^2']]
table.to_csv(RES_DIR / 'm3_final_error_table.csv', float_format='%.4f')
display(table.round(4))
print(f"std of FM = {y.std(ddof=1):.4f}; worst-pt numbers refer to this table:")
display(df[['D_m', 'T_N', 'DL', 'FM', 'n', 'batch']].round(3))

Observation (do the RMSE differences between the RBFs and the GP exceed the 5-fold ± std? is the GP's uncertainty
honest (mean z² ≈ 1)?): _..._

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), sharex=True, sharey=True)
lo_, hi_ = y.min() - 0.04, y.max() + 0.04
for ax, (name, (mu, sd)) in zip(axes, loo.items()):
    if np.all(np.isfinite(sd)):
        ax.errorbar(y, mu, yerr=2 * sd, fmt='none', ecolor='grey', alpha=0.5)
    sc = ax.scatter(y, mu, c=df.DL, norm=LogNorm(), cmap='plasma', edgecolor='k', s=45, zorder=3)
    for i in df.index:
        if abs(y[i-1] - mu[i-1]) > 0.04:
            ax.annotate(str(i), (y[i-1], mu[i-1]), xytext=(4, 3), textcoords='offset points', fontsize=8)
    ax.plot([lo_, hi_], [lo_, hi_], 'k--', lw=1)
    r = table.loc[name]
    ax.set_title(f"{name}\nLOO {r['LOO RMSE']:.3f} | 5-fold {r['5-fold RMSE mean']:.3f}±{r['5-fold RMSE std']:.3f}",
                 fontsize=9)
    ax.set_xlabel('Oracle FM'); ax.grid(alpha=0.3); ax.set_xlim(lo_, hi_); ax.set_ylim(lo_, hi_)
axes[0].set_ylabel('LOO prediction')
fig.colorbar(sc, ax=axes, label='DL (N/m²)', shrink=0.85)
fig.suptitle('M3 verification, final (30 runs, 29 unique points): LOO parity, RMSE in FM units')
fig.savefig(FIG_DIR / 'm3_loo_parity_final.png', dpi=150, bbox_inches='tight'); plt.show()

Observation: _..._

## 3. Final surrogate with uncertainty

The cubic RBF has no built-in uncertainty, so we use a **jackknife band**: refit 29 times leaving one point out and
measure how much the prediction moves (`jackknife_band`). Shown next to the GP's σ for comparison.

Prediction (where is the band widest?): _..._

In [ ]:
rbf = RBFSurrogate('cubic', 0.0).fit(X, y)
gp = GPSurrogate().fit(X, y)
Dg, Tg = np.meshgrid(np.linspace(D_MIN, D_MAX, 70), np.linspace(T_MIN, T_MAX, 70))
Xg = make_features(Dg.ravel(), Tg.ravel())
F, jk = jackknife_band(RBFSurrogate('cubic', 0.0), X, y, Xg)
F, jk = F.reshape(Dg.shape), jk.reshape(Dg.shape)
DLg = disk_loading(Dg, Tg)
m1 = pd.read_csv(RES_DIR / 'm1_fm_designs.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, Z, title, cmap in [(axes[0], F, 'Final FM surrogate: cubic RBF (29 points)', 'viridis'),
                           (axes[1], jk, 'Its uncertainty: jackknife std', 'magma')]:
    cf = ax.contourf(Dg, Tg, Z, levels=20, cmap=cmap); plt.colorbar(cf, ax=ax)
    cs = ax.contour(Dg, Tg, DLg, levels=[25, 50, 100, 150, 217, 300], colors='w', linewidths=0.7, linestyles='--')
    ax.clabel(cs, fmt='DL=%d', fontsize=7)
    for b, mk in [('result_1', 'o'), ('result_2', 's'), ('result_3', '^')]:
        s = df[df.batch.str.contains(b)]
        ax.scatter(s.D_m, s.T_N, marker=mk, c='w', edgecolor='k', s=45, zorder=3, label=b)
    for _, r in m1.iterrows():
        ax.plot([r.D, r.D], [r.T_hover_empty, r.T_hover_loaded], '-', color='cyan', lw=2.5)
        ax.annotate(str(r.design).split()[0], (r.D, r.T_hover_loaded), xytext=(-10, 3),
                    textcoords='offset points', color='cyan', fontweight='bold')
    ax.set_xlabel('D (m)'); ax.set_ylabel('T (N)'); ax.set_title(title)
axes[0].legend(loc='lower right', fontsize=8)
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_final_surrogate.png', dpi=150); plt.show()

T_line = np.linspace(T_MIN, T_MAX, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for ax, Ds in zip(axes, [0.36, 0.46, 0.69]):
    Xs = make_features(np.full_like(T_line, Ds), T_line)
    f, s = jackknife_band(RBFSurrogate('cubic', 0.0), X, y, Xs)
    mg, sg = gp.predict(Xs, return_std=True)
    ax.fill_between(T_line, f - 2*s, f + 2*s, color='green', alpha=0.2, label='RBF cubic ±2 jackknife std')
    ax.plot(T_line, f, 'g', label='RBF cubic (final)')
    ax.plot(T_line, mg, 'r--', lw=1, label='GP (D, T)')
    near = np.abs(df.D_m - Ds) < 0.04
    ax.scatter(df.T_N[near], df.FM[near], c='k', s=40, zorder=5, label='data |ΔD|<0.04')
    ax.set_title(f'D = {Ds} m'); ax.set_xlabel('T (N)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('FM'); axes[0].set_ylim(0.1, 0.8); axes[0].legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_final_slices.png', dpi=150); plt.show()

Observation: _..._

## 4. M1 re-run with the final surrogate: how far did the optimum move?

`run_all.py` re-runs M1 with the cubic RBF (`m1_vehicle/fm_rerun.py`; hover points constrained to the convex hull of
the oracle points; plate mass from M2). Compared here: FM0 = 0.65, the cubic RBF fitted on 24 runs, the final fit on
30 runs, and, as a robustness check, two other surrogates fitted on 30 runs
(`results/m3_m1_surrogate_sensitivity.csv`, all with the same M2 plate mass).

Prediction: _..._

In [ ]:
cmp_all = pd.read_csv(RES_DIR / 'm1_fm_comparison.csv').set_index('design')       # written by run_all (final fit)
sens = pd.read_csv(RES_DIR / 'm3_m1_surrogate_sensitivity.csv')
cub = sens[sens.surrogate == 'RBF cubic (chosen)']
s24 = cub[cub.fit == '24 runs'].set_index('design'); s30 = cub[cub.fit == '30 runs'].set_index('design')
cmp = pd.DataFrame({
    'm FM0 (kg)': cmp_all.m_FM0, 'm 24-run fit (kg)': s24.m_continuous, 'm final (kg)': s30.m_continuous,
    'Δm vs FM0 (%)': 100 * (s30.m_continuous / cmp_all.m_FM0 - 1),
    'Δm vs 24-run (%)': 100 * (s30.m_continuous / s24.m_continuous - 1),
    'D FM0 (m)': cmp_all.D_FM0, 'D 24-run': s24.D_continuous, 'D final': s30.D_continuous,
    'FM loaded / empty': s30.FM_loaded.round(3).astype(str) + ' / ' + s30.FM_empty.round(3).astype(str),
    'R of FM0 design with real FM (km)': cmp_all.R_of_FM0_design_with_surr_FM,
    'snapped FM0 → final (kg)': cmp_all.m_snapped_FM0.round(3).astype(str) + ' → ' + cmp_all.m_snapped_surr.round(3).astype(str)})
cmp.to_csv(RES_DIR / 'm3_m1_optimum_shift.csv', float_format='%.4f')
display(cmp.round(3))

fin = sens[sens.fit == '30 runs']
display(fin.pivot(index='design', columns='surrogate', values='m_continuous').round(3))
display(fin.pivot(index='design', columns='surrogate', values='m_snapped').round(3))

Observation (how much did the optimum move from FM0? from the 24-point fit? does the choice of surrogate matter
for the designs?): _..._

## 5. Notes for the report (draft, rewrite in your own words)

**Noise.** _..._

**What a smooth model handles badly, and how it was dealt with.** _..._

**Final choice of surrogate, and why.** _..._